In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!pip install transformers datasets tensorflow pdfplumber --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 135.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 139.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 121.2 MB/s eta 0:00:00


In [ ]:
!pip install keras-hub

In [ ]:
from datasets import load_dataset

In [ ]:
dataset=load_dataset("abisee/cnn_dailymail","3.0.0")

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [ ]:
print(dataset["train"][0]["article"][:500])
print("\n SUMMARY")
print(dataset["train"][0]["highlights"])

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s

 SUMMARY
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [ ]:
small_train=dataset["train"].select(range(500))
small_val=dataset["validation"].select(range(100))
print(small_train)
print(small_val)

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 500
})
Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 100
})


In [ ]:
import keras_hub
preprocessor=keras_hub.models.BartSeq2SeqLMPreprocessor.from_preset(
    "bart_base_en",
    encoder_sequence_length=1024,
    decoder_sequence_length=128
)

100%|██████████| 430/430 [00:00<00:00, 450kB/s]


100%|██████████| 649/649 [00:00<00:00, 1.16MB/s]


100%|██████████| 0.99M/0.99M [00:01<00:00, 778kB/s]


100%|██████████| 446k/446k [00:01<00:00, 427kB/s]


In [ ]:
model=keras_hub.models.BartSeq2SeqLM.from_preset(
    "bart_base_en",
    preprocessor=preprocessor
)

In [ ]:
def to_dictionary_format(hf_dataset):
  return {
      "encoder_text" : list(hf_dataset["article"])
      ,"decoder_text" : list(hf_dataset["highlights"])
  }

In [ ]:
train_dict=to_dictionary_format(small_train)
val_dict=to_dictionary_format(small_val)

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices(train_dict).batch(4)
val_ds = tf.data.Dataset.from_tensor_slices(val_dict).batch(4)

print(train_ds)
print(val_ds)

<_BatchDataset element_spec={'encoder_text': TensorSpec(shape=(None,), dtype=tf.string, name=None), 'decoder_text': TensorSpec(shape=(None,), dtype=tf.string, name=None)}>
<_BatchDataset element_spec={'encoder_text': TensorSpec(shape=(None,), dtype=tf.string, name=None), 'decoder_text': TensorSpec(shape=(None,), dtype=tf.string, name=None)}>


In [ ]:
model.compile(
    weighted_metrics=["accuracy"],
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5)
)

In [ ]:
model.fit(
    train_ds,
    epochs=1,
    validation_data=val_ds
)

125/125 ━━━━━━━━━━━━━━━━━━━━ 218s 845ms/step - accuracy: 0.5379 - loss: 1.0824 - val_accuracy: 0.5619 - val_loss: 0.7519


In [ ]:
sample_article=dataset["test"][0]["article"]
output=model.generate(
    {"encoder_text" : sample_article , "decoder_text" : ""},
    max_length=128
)
print("SUmmary \n")
print(output)

SUmmary 

NEW: Palestinian Authority becomes the 123rd member of the ICC .
The ICC opens its preliminary investigation into the situation in Palestinian territories .
The new seat is in The Hague, in the Netherlands .
It's the first time the court has opened a preliminary examination into the situation in Palestinian territories .


In [ ]:
print(dataset["test"][0]["highlights"])

Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .
